# Demo: FP-Growth* — Khai thác Tập Phổ Biến

Notebook minh họa cách dùng cài đặt FP-Growth\* của nhóm: đọc dữ liệu định dạng SPMF, khai phá tập phổ biến, in FP-tree, xử lý trường hợp đặc biệt (single-path), và một ví dụ ứng dụng (sinh luật kết hợp).

> **Trước khi nộp:** chọn *Kernel → Restart & Run All* để đảm bảo toàn bộ notebook chạy sạch từ đầu.

## 1. Nạp module

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # kích hoạt môi trường Project.toml
include(joinpath(@__DIR__, "..", "src", "FPGrowthStar.jl"))
using .FPGrowthStar
using Random
Random.seed!(42)   # cố định seed để kết quả tái lập được

## 2. Đọc dữ liệu & khai phá tập phổ biến

`example1.txt` là CSDL đồ chơi (7 giao dịch). Mỗi dòng là một giao dịch gồm các item (số nguyên) cách nhau bởi khoảng trắng — đúng định dạng SPMF.

In [ ]:
transactions = read_spmf(joinpath(@__DIR__, "..", "data", "toy", "example1.txt"))
println("Số giao dịch: ", length(transactions))
transactions

In [ ]:
minsup = 3
results = fpgrowth_star(transactions, minsup)
println("Số tập phổ biến (minsup=$minsup): ", length(results))
write_results(stdout, results)

## 3. Trực quan hóa FP-tree

Cấu trúc dữ liệu cốt lõi của thuật toán. `build_fptree` dựng cây nén từ toàn bộ CSDL.

In [ ]:
tree = build_fptree(transactions, minsup)
println("Thứ tự item (giảm dần theo support): ", tree.ordered_items)
println()
print_tree(tree.root)

## 4. Trường hợp đặc biệt: FP-tree nhánh đơn (single-path)

Ví dụ 2 (`example2.txt`) là tình huống đặc biệt **single-path**: FP-tree chỉ có một nhánh. Khi đó FP-Growth\* không đệ quy nữa mà **liệt kê trực tiếp** mọi tập con khác rỗng của nhánh — đúng $2^n-1$ tập phổ biến (với $n$ là số item trên nhánh).

In [ ]:
t2 = read_spmf(joinpath(@__DIR__, "..", "data", "toy", "example2.txt"))
r_star = fpgrowth_star(t2, 1)
n_items = length(unique(vcat(t2...)))
println("Số item phân biệt: ", n_items)
println("FP-Growth* tìm được: ", length(r_star), " tập phổ biến")
println("Kỳ vọng 2^n - 1   : ", 2^n_items - 1)
println("Khớp? ", length(r_star) == 2^n_items - 1)

## 5. Ứng dụng: sinh luật kết hợp (Market Basket Analysis)

Từ tập phổ biến, hàm `association_rules` của nhóm (`src/rules.jl`) sinh luật $X \Rightarrow Y$ với **support**, **confidence** và **lift** — tất cả từ chính cài đặt của nhóm, không dùng thư viện ngoài.

Dưới đây minh họa trên CSDL đồ chơi. Ứng dụng đầy đủ trên CSDL bán lẻ thực (Retail, 88K giao dịch → ~6000 luật) xem `test/test_rules.jl`, kết quả xuất ra `results/rules_retail.csv`.

In [ ]:
rules = association_rules(results, length(transactions); minconf=0.6)
println("Số luật (minconf=0.6): ", length(rules))
println("\nTop luật kết hợp (theo lift):")
for r in first(rules, min(10, length(rules)))
    println("  ", r.antecedent, " => ", r.consequent,
            "  (sup=", r.support,
            ", conf=", round(r.confidence, digits=2),
            ", lift=", round(r.lift, digits=2), ")")
end